# Food Price Watch 06: view results and download
Reads the finished exports on Drive. Run after notebook 05. Nothing is computed or changed here. The files hold TCB prices: keep them private and out of git.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fresh clone of main (or fast-forward if this runtime already has it).
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
%cd /content/foodprice_watch
%env PYTHONPATH=/content/foodprice_watch/src
%env FPW_ARTIFACT_ROOT=/content/drive/MyDrive/FoodPriceWatch
from pathlib import Path
import csv, hashlib, json, shutil
ROOT = Path('/content/drive/MyDrive/FoodPriceWatch')
for folder in ['inbox/project', 'inbox/external', 'data/processed', 'outputs', 'reports/tables',
               'reports/figures', 'reports/project/tables', 'state', 'config', 'logs', 'models']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
# First run only: copy the starting state files into Drive. Existing files and seals are never replaced.
for name in ['state/stages.json', 'config/item_driver_map.yaml', 'DECISIONS.md', 'PROGRESS.md', 'DEVIATIONS.md',
             'reports/project/tables/P2_audit.csv', 'reports/project/tables/P2_audit_summary.csv']:
    if not (ROOT / name).exists():
        shutil.copy2(name, ROOT / name)
# Remove half-written files left by an interrupted session.
for folder in ['outputs', 'data']:
    for tmp in (ROOT / folder).rglob('*.tmp'):
        tmp.unlink()
CODE_COMMIT = !git rev-parse HEAD
CODE_COMMIT = CODE_COMMIT[0]
CHECKS_PASS = False
stage_ok = {}

In [ ]:
CHECKS_PASS = False
stage_ok.clear()
!python -m fpw.check
CHECKS_PASS = get_ipython().user_ns['_exit_code'] == 0
assert CHECKS_PASS, 'Startup check failed; fix it before running stages'

## View

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
EX = ROOT / 'outputs/export'
fc = pd.read_parquet(EX / 'forecasts.parquet'); al = pd.read_csv(EX / 'alerts.csv', parse_dates=['tdate']); dr = pd.read_csv(EX / 'alert_drivers.csv')
ITEM = sorted(fc.item_id.unique())[0]    # change to any item name from the list printed below
print(len(fc.item_id.unique()), 'items; origin', fc.origin_date.max().date(), '; items:', *sorted(fc.item_id.unique())[:6], '...')
d = fc[fc.item_id == ITEM].sort_values('horizon'); print('Item:', ITEM, '| last price', d.last_price.iloc[0]); display(d[['horizon', 'forecast_low', 'forecast', 'forecast_high']].round(1))
recent = al[al.tdate >= al.tdate.max() - pd.Timedelta(days=30)]; print(len(recent), 'alert starts in the last 30 days of the data'); display(recent.sort_values('tdate', ascending=False).head(15).round(3))
g = [c for c in ['price_history', 'calendar', 'weather', 'onion_export_measures', 'exchange_rate'] if c in dr.columns]
ax = dr[g].abs().mean().plot.bar(title='Average size of each driver group on alert days'); plt.tight_layout(); plt.show()


## Download to this computer
Sends two zip files to your browser's Downloads folder: the three export files (for the local dashboard) and the report tables, figures and results register. The browser may ask to allow multiple downloads.

In [ ]:
from google.colab import files
import shutil
for name, src in [('export_files', ROOT / 'outputs/export'), ('report_tables_and_figures', ROOT / 'reports/project')]:
    z = shutil.make_archive('/content/' + name, 'zip', src); print(name, '->', z); files.download(z)
